# Januário Leads — Fase 9 V6.3: sessão estendida com retomada segura

**Corrige o encerramento prematuro da V6.2 por limite de 320 tentativas**, sem apagar os dados existentes. Na execução enviada, a V6.2 terminou com **0 erros**, 320 lotes/tentativas (308 lotes OK), **333.418 registros totais** e **276.269 empresas úteis**. A causa da parada foi `MAX_DOWNLOADS_PER_SESSION=320`, não uma falha do Overture.

## V6.3 — ajustes

- Até **1.200 tentativas de lote** em cada sessão, em vez de 320, **ou até 9 horas**, o que acontecer primeiro. O número é um limite de segurança interno, não um direito de realizar 1.200 downloads nem uma garantia de disponibilidade.
- **Sem espera fixa de 90 segundos** entre lotes bem-sucedidos. A próxima consulta começa após gravar e conferir os dados da anterior. Não há paralelismo.
- O **erro de taxa da fonte (`fonte_taxa`) interrompe imediatamente as consultas**, com bloqueio conservador de 8 horas para reiniciar. Erros transitórios de rede permitem até 2 tentativas com intervalos apenas após falhas.
- São preservados: proteção contra duas Colabs simultâneas (checagem de checkpoints recentes de 12 minutos), uso **exclusivo da HOMOLOGAÇÃO**, máximo de 40 novos registros por município/download, até 3 municípios próximos, cidades sem novos candidatos suspensas por 14 dias, teto de 340 MiB Overture e 440 MiB para o banco completo, meta de 600.000 empresas úteis, integridade dos checkpoints e não sobrescrita de empresas já salvas.
- Uma sessão do **Google Colab gratuito pode ser encerrada antes das 9 horas**, independente do código. Não é possível garantir execução ininterrupta.

## Credenciais — regra de segurança atualizada

**Esta versão não aceita chaves JWT legadas `eyJ...` (incluindo `service_role`).** A célula 3 só permite uma nova credencial moderna `sb_secret_...` da **homologação**, digitada no campo oculto; nenhuma chave deve ser salva no notebook ou compartilhada. **Importações continuam suspensas até que a antiga chave comprometida tenha sido efetivamente revogada e os consumidores testados.** O restante do código legado usa nomes de variáveis históricos, mas é a verificação da célula 3 que bloqueia credenciais antigas antes de abrir a conexão.

## Para executar

1. **Pare e desconecte a Colab V6.2 antiga**. Não rode duas versões ao mesmo tempo. Respeite **12 minutos desde o último checkpoint** antes da nova execução.
2. Abra esta V6.3 no Colab com runtime padrão de CPU. Execute as células 1, 2 e 3 em ordem (ou **Executar tudo**).
3. Digite **RETOMAR_V6** para confirmar importação **somente na HOMOLOGAÇÃO**.
4. Cole sua chave secreta apenas no **campo oculto** quando solicitado. Jamais grave chaves em células, GitHub, arquivos compartilhados, logs ou chats.
5. Confira `ANTES V6.3`, `LOTE`, `CITY_RESULT`, `BATCH_SUMMARY`, `RESUMO` e `FINAL`. A execução busca o progresso real no Supabase e retoma a fila restante; não começa do zero.

## Observação sobre a parada anterior

O último log V6.2 apresentava `PARADA SEGURA: limite de downloads ou duração máxima da sessão atingido` exatamente após o **lote 320**, sem erros e abaixo dos limites de armazenamento. Portanto, não há motivo para esperar um cooldown por taxa da fonte por causa desta parada específica. **A checagem de atividade recente (12 minutos) permanece obrigatória.**


In [ ]:
# 1 — Instalar dependências necessárias somente quando executar
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências instaladas. Execute a próxima célula.")

In [ ]:
# 2 — Planejamento: baixar scripts do repositório público e população IBGE (SEM chaves)
import collections, datetime as dt, getpass, json, math, os, random, re, subprocess, sys, time, urllib.parse, urllib.request
from pathlib import Path
BASE="https://raw.githubusercontent.com/wesleyshoto702-hub/januarioleads/e88df68d6350c5d0e97210bd1d495e76f89be09e/scripts/"
STAGING_URL="https://prlrxjothrslhswpkoza.supabase.co"
WORK=Path("/content"); WORK.mkdir(exist_ok=True)
for filename in ("expand_overture_to_100_staging.py","expand_overture_600k_staging.py"):
    url=BASE+filename
    req=urllib.request.Request(url,headers={"User-Agent":"JanuarioLeads-Import/1.0"})
    with urllib.request.urlopen(req,timeout=40) as response:
        source=response.read(120000)
    if len(source)>90000 or not source.startswith(b"#!/usr/bin/env python3"):
        raise RuntimeError("Código do importador inválido; cancelado sem consultar o Overture.")
    (WORK/filename).write_bytes(source)
sys.path.insert(0,str(WORK))
import importlib
import expand_overture_to_100_staging as v4
import expand_overture_600k_staging as v5
# Re-executing Cell 2 must use the latest GitHub files, not Python's
# module cache from a previous failed Colab attempt.
v4=importlib.reload(v4)
v5=importlib.reload(v5)
assert all(hasattr(v4,name) for name in ("normalize","api","ensure_state_boundaries",
             "municipalities_index","batch_bbox_allowed","place_record","same_business"))
assert all(hasattr(v5,name) for name in ("main","safe_capacity","safe_error_kind",
             "record_checkpoint","existing_city","current_stats"))
assert v5.STAGING_URL==STAGING_URL and v5.GOAL_COMPANIES==600000
assert v5.OVERTURE_MAX_MIB==340 and v5.DATABASE_MAX_MIB==440

# População de referência 2022: cópia estática no GitHub do próprio projeto.
# Evita a API do IBGE, que pode não responder no Colab; fonte documentada:
# https://github.com/lsbastos/popBR_mun/blob/master/poptcu2010-2022_2024.csv
# A coluna POP22 é uma referência de população, NÃO o Censo 2022.
# É usada somente para agrupar as cidades pelo porte e definir metas conservadoras.
POP_SNAPSHOT_URL=("https://raw.githubusercontent.com/wesleyshoto702-hub/"
                  "januarioleads/e88df68d6350c5d0e97210bd1d495e76f89be09e/data/populacao-municipal-2022-referencia.json")

def population_index():
    req=urllib.request.Request(
        POP_SNAPSHOT_URL,
        headers={"User-Agent":"JanuarioLeads-Population-Reference/1.0",
                 "Accept":"application/json"})
    # A cópia tem ~130 kB. A API IBGE não é chamada em momento algum.
    for timeout in (15,25):
        try:
            with urllib.request.urlopen(req,timeout=timeout) as response:
                raw=response.read(250001)
            if len(raw)>250000:
                raise ValueError("Arquivo de população excedeu tamanho esperado")
            doc=json.loads(raw.decode("utf-8"))
            meta=doc.get("metadata") or {}
            entries=doc.get("populations") or {}
            if meta.get("reference_year")!=2022 or not isinstance(entries,dict):
                raise ValueError("Metadados de população inválidos")
            result={}
            for label,value in entries.items():
                if not isinstance(label,str) or "|" not in label:
                    raise ValueError("Código de município inválido no arquivo")
                uf,city_key=label.split("|",1)
                if (not re.fullmatch(r"[A-Z]{2}",uf) or
                    not city_key or
                    not isinstance(value,int) or isinstance(value,bool) or value<=0):
                    raise ValueError("População municipal inválida")
                result[(uf,city_key)]=value
            if len(result)<5500:
                raise ValueError("Referência municipal incompleta")
            return result
        except (urllib.error.URLError, TimeoutError, OSError):
            if timeout==15:
                print("Referência populacional indisponível. Nova tentativa controlada...")
                continue
            raise RuntimeError("Não foi possível carregar a referência populacional no GitHub. Reexecute a célula 2.")
    raise RuntimeError("Não foi possível carregar a população municipal.")

try:
    POPULATIONS=population_index()
    print("População de referência 2022 carregada:",len(POPULATIONS),"municípios.",
          "Sem dependência da API IBGE.")
except (ValueError,RuntimeError,UnicodeError,json.JSONDecodeError) as exc:
    POPULATIONS={}
    print("PARADA SEGURA:",str(exc),
          "Nenhuma importação poderá começar sem a população municipal.")


def plan_for_city(row):
    population=POPULATIONS.get((row["state"],row["city_key"]))
    if population is None:tier="unknown"
    elif population<30000:tier="small"
    elif population<150000:tier="medium"
    elif population<700000:tier="large"
    else:tier="metro"
    return {"tier":tier,"target":v5.CITY_CAPS[tier],"population":population}

def read_stage(secret,path):
    return v4.api(STAGING_URL,secret,path)

def paged_stage(secret,table,fields):
    output=[]; offset=0
    while True:
        qs=urllib.parse.urlencode({"select":fields,"order":"state.asc,city_key.asc",
                                   "limit":"1000","offset":str(offset)})
        page=read_stage(secret,"/rest/v1/"+table+"?"+qs)
        if not isinstance(page,list):
            raise RuntimeError("Falha na leitura dos dados de homologação")
        output.extend(page)
        if len(page)<1000:break
        offset+=len(page)
    return output

def priority(row):
    level={"metro":4,"large":3,"medium":2,"small":1,"unknown":0}[row["plan"]["tier"]]
    # Prioritize high-population cities with few niche groups represented.
    niches=sum(int(row.get(k,0) or 0)==0 for k in (
        "barbers","food","fitness","beauty","health","auto_services","lodging","retail"))
    return (level,niches,-int(row["imported_rows"]),row["plan"]["target"]-int(row["imported_rows"]))

def ordered_queue(rows,blocked):
    by_state=collections.defaultdict(list)
    for row in rows:
        key=(row["state"],row["city_key"])
        if key in blocked:continue
        row["plan"]=plan_for_city(row)
        if int(row["imported_rows"])>=row["plan"]["target"]:continue
        by_state[row["state"]].append(row)
    for uf in by_state:
        by_state[uf].sort(key=priority,reverse=True)
    ordered=[]
    while any(by_state.values()):
        for state in sorted(by_state):
            if by_state[state]:ordered.append(by_state[state].pop(0))
    return ordered

def close_groups(rows):
    groups=[];unused={(r["state"],r["city_key"]) for r in rows}
    by_state=collections.defaultdict(list)
    for row in rows:by_state[row["state"]].append(row)
    for row in rows:
        key=(row["state"],row["city_key"])
        if key not in unused:continue
        unused.remove(key);group=[row]
        for other in by_state[row["state"]]:
            k=(other["state"],other["city_key"])
            if k not in unused:continue
            if all(v4.normalize(str(other["city"]))!=v4.normalize(str(x["city"])) and
                   distance_km(other,x)<=25 for x in group):
                group.append(other);unused.remove(k)
                if len(group)>=3:break
        groups.append(group)
    return groups

def distance_km(a,b):
    try:
        x1,y1=float(a["center_lat"]),float(a["center_lon"])
        x2,y2=float(b["center_lat"]),float(b["center_lon"])
        h=math.sin(math.radians(x2-x1)/2)**2+math.cos(math.radians(x1))*math.cos(math.radians(x2))*math.sin(math.radians(y2-y1)/2)**2
        return 12742*math.asin(min(1,math.sqrt(h)))
    except (ValueError,TypeError,KeyError):return float("inf")



# V6: priority scheduling only. Does not change fetch frequency, caps or importer.
# Grouping still calls the V5 geographic safety checks through close_groups().
def optimized_groups(rows):
    raw=close_groups(rows)
    by_state=collections.defaultdict(list)
    for group in raw:
        assert 1<=len(group)<=3
        assert len({r["state"] for r in group})==1
        state=group[0]["state"]
        # A group of 3 can amortize the same 90s source cooldown over
        # 3 towns. Within each state, prefer groups with the most room left.
        room=sum(min(v5.MAX_NEW_PER_BATCH,
                     max(0,int(r["plan"]["target"])-int(r["imported_rows"])))
                 for r in group)
        shortage=sum(sum(int(r.get(k,0) or 0)==0 for k in (
            "barbers","food","fitness","beauty","health",
            "auto_services","lodging","retail")) for r in group)
        tier=sum({"metro":4,"large":3,"medium":2,"small":1,"unknown":0}[
            r["plan"]["tier"]] for r in group)
        by_state[state].append(((len(group),room,shortage,tier),group))
    for state in by_state:
        by_state[state].sort(key=lambda item:item[0],reverse=True)
    output=[]
    # Fair round-robin: do not starve any Brazilian UF by greedily visiting
    # only a few large cities; high-yield groups lead within each UF.
    while any(by_state.values()):
        for state in sorted(by_state):
            if by_state[state]:
                output.append(by_state[state].pop(0)[1])
    assert sum(map(len,output))==len(rows)
    assert len({(r["state"],r["city_key"]) for g in output for r in g})==len(rows)
    return output


# Diagnóstico V5 independente da Fase 9 V4 — sem imprimir stdout/stderr
# remoto, endereços completos ou chaves. Nenhuma consulta extra ao Overture.
V5_KNOWN_ERRORS={
    "fonte_taxa","fonte_acesso","fonte_download","supabase_taxa",
    "supabase_auth","limite_banco","dependencia_ausente","malha_ibge",
    "erro_codigo","erro_dados","erro_configuracao","rede_conexao",
    "rede_timeout","timeout_subprocesso","batch_largura",
    "fonte_indisponivel","supabase_indisponivel","erro_desconhecido",
}
def classify_v5_error(output,returncode=None):
    message=str(output or "").lower()[-20000:]
    stages=re.findall(r"v5_error_stage=([a-z_]{1,40})",message)
    if stages and stages[-1]=="municipal_boundaries":
        return "malha_ibge"
    # Rate limit from Overture CLI stdout/stderr takes precedence over generic
    # CalledProcessError, so its global 8h cooldown is preserved.
    if "supabase api 429" in message:
        return "supabase_taxa"
    if any(s in message for s in ("http error 429","http 429","too many requests",
                                  "rate limit exceeded","throttled","throttling")):
        return "fonte_taxa"
    match=re.findall(r"v5_error_class=([a-z0-9_]{1,40})",message)
    if match and match[-1] in V5_KNOWN_ERRORS:
        return match[-1]
    if "supabase api 401" in message or "supabase api 403" in message:
        return "supabase_auth"
    if "http error 403" in message or "http error 401" in message:
        return "fonte_acesso"
    if "batch_too_wide" in message:
        return "batch_largura"
    if any(s in message for s in ("database safety threshold","database reached",
                                  "safe stop: storage","quota exceeded")):
        return "limite_banco"
    if any(s in message for s in ("429 response from supabase","supabase throttling")):
        return "supabase_taxa"
    if any(s in message for s in ("module not found","modulenotfounderror",
                                  "no module named","filenotfounderror",
                                  "command not found")):
        return "dependencia_ausente"
    if "attributeerror" in message or "importerror" in message:
        return "erro_codigo"
    if "typeerror" in message or "keyerror" in message or "valueerror" in message:
        return "erro_dados"
    if "supabase api" in message and any(s in message for s in (" 500"," 502"," 503"," 504")):
        return "supabase_indisponivel"
    if any(s in message for s in ("http error 502","http error 503","http error 504")):
        return "fonte_indisponivel"
    if any(s in message for s in ("connection reset","connection refused",
                                  "name resolution","gaierror","network is unreachable")):
        return "rede_conexao"
    if any(s in message for s in ("timed out","timeouterror","readtimeout","connecttimeout")):
        return "rede_timeout"
    if "ibge" in message or "boundary" in message or "boundaries" in message:
        return "malha_ibge"
    if "calledprocesserror" in message or "overturemaps download" in message:
        return "fonte_download"
    return "erro_desconhecido"

def stop_for_v5_error(code):
    """Avoid retry storms after throttling, uncertain source errors or coding errors."""
    return code in (
       "fonte_taxa","fonte_acesso","fonte_download","supabase_taxa",
       "supabase_auth","limite_banco","dependencia_ausente",
       "erro_codigo","erro_dados","erro_configuracao",
       "malha_ibge","fonte_indisponivel","supabase_indisponivel",
       "erro_desconhecido","batch_largura"
    )


def recent_rate_limit(secret):
    times=[]
    for table in ("overture_phase9_attempts","overture_phase9_v5_attempts"):
        q=urllib.parse.urlencode({"select":"attempted_at","error_code":"eq.fonte_taxa",
                                  "order":"attempted_at.desc","limit":"1"})
        res=read_stage(secret,"/rest/v1/"+table+"?"+q)
        if res:times.append(dt.datetime.fromisoformat(res[0]["attempted_at"].replace("Z","+00:00")))
    return max(times) if times else None

# Sessão livre: sem horário fixo do dia, apenas um máximo de 9 horas
# e os demais limites da fonte, quota e banco.
MIN_INTERVAL=0
COOLDOWN_HOURS=8
MAX_DOWNLOADS_PER_SESSION=1200
MAX_RUN_SECONDS=9*3600
assert MIN_INTERVAL==0 and COOLDOWN_HOURS>=6
assert v5.MAX_NEW_PER_BATCH==40 and v5.MAX_CITIES_PER_BATCH==3
# Validação offline e determinística do classificador e dos limites essenciais.
assert classify_v5_error("V5_ERROR_STAGE=source_download\\nV5_ERROR_CLASS=fonte_taxa",1)=="fonte_taxa"
assert classify_v5_error("V5_ERROR_STAGE=source_download\\nV5_ERROR_CLASS=rede_timeout",1)=="rede_timeout"
assert classify_v5_error("V5_ERROR_STAGE=storage_guard\\nV5_ERROR_CLASS=supabase_auth",1)=="supabase_auth"
assert MAX_RUN_SECONDS==9*3600 and MAX_DOWNLOADS_PER_SESSION==1200
print("TESTE LOCAL V6.3 OK: sem pausa fixa, classificação e limites validados. Nenhum download de Overture neste teste.")


In [ ]:
# 3 — EXECUÇÃO EXPLÍCITA: confirmação e credencial oculta da HOMOLOGAÇÃO
# Não importa se a célula 2 não estiver carregada: o teste ao fim evita NameError.
import datetime as dt

def ampliar_600k():
    print("FASE 9 V6.3 — retomada resiliente (até 9 horas por sessão).")
    print("Meta: 600.000 empresas úteis; Overture <=340 MiB; banco <=440 MiB.")
    print("Máx. 40 novos por cidade/download; até 3 cidades vizinhas; sem espera fixa entre lotes.")
    if len(POPULATIONS)<5500:
        print("PARADA SEGURA: dados de população não carregados. Execute a célula 2 novamente.")
        return
    confirm=input("Para importar SOMENTE na HOMOLOGAÇÃO, digite RETOMAR_V6: ").strip()
    if confirm!="RETOMAR_V6":
        print("CANCELADO. Não houve importação.")
        return
    secret=getpass.getpass("Chave SECRETA da HOMOLOGAÇÃO (oculta): ").strip()
    if not secret.startswith("sb_secret_"):
        print("BLOQUEADO: use somente sb_secret_ moderna da HOMOLOGAÇÃO, após concluir a rotação da Fase 0.1. Não compartilhe a chave.")
        return
    try:
        last=recent_rate_limit(secret)
        now=dt.datetime.now(dt.timezone.utc)
        if last and now<last+dt.timedelta(hours=COOLDOWN_HOURS):
            print("PARADA SEGURA: respeitar o Overture até",
                  (last+dt.timedelta(hours=COOLDOWN_HOURS)).isoformat())
            return
        stats=v5.current_stats(STAGING_URL,secret)
        print("ANTES V6.3: registros",stats["total_companies"],
              "| Empresas úteis:",stats["useful_companies"],
              "| Overture MiB:",round(stats["overture_bytes"]/v5.MIB,2),
              "| Banco MiB:",round(stats["database_bytes"]/v5.MIB,2))
        if v5.safe_capacity(stats,1)==0:
            print("PARADA: meta nacional ou espaço insuficiente")
            return
        gaps=paged_stage(secret,"overture_phase9_v5_gaps",
          "state,city,city_key,imported_rows,barbers,food,fitness,beauty,health,auto_services,lodging,retail,with_phone,with_email,with_website")
        centers=paged_stage(secret,"overture_phase9_city_centers",
          "state,city_key,center_lat,center_lon")
        loc={(x["state"],x["city_key"]):x for x in centers}
        for row in gaps:
            center=loc.get((row["state"],row["city_key"]),{})
            row.update(center_lat=center.get("center_lat"),center_lon=center.get("center_lon"))
        checkpoints=paged_stage(secret,"overture_phase9_v5_attempts",
          "state,city_key,result,attempted_at,error_code")
        # No-more cities are postponed 14 days, avoiding repeated expensive
        # downloads for source data that had no suitable new companies.
        blocked=set()
        for c in checkpoints:
            if c["result"]!="no_more":continue
            attempted=dt.datetime.fromisoformat(c["attempted_at"].replace("Z","+00:00"))
            if dt.datetime.now(dt.timezone.utc)-attempted<dt.timedelta(days=14):
                blocked.add((c["state"],c["city_key"]))
        # Safety interlock: prevents a second session starting right after a V5
        # checkpoint; it is not a perfect distributed process lock.
        last_checkpoint=max((
            dt.datetime.fromisoformat(c["attempted_at"].replace("Z","+00:00"))
            for c in checkpoints
        ),default=None)
        if last_checkpoint and (dt.datetime.now(dt.timezone.utc)-
                                last_checkpoint)<dt.timedelta(minutes=12):
            print("PARADA SEGURA V6: checkpoint recente em",
                  last_checkpoint.isoformat(),
                  "| aguarde 12 minutos sem novos checkpoints e encerre a Colab antiga.")
            return
        targets=ordered_queue(gaps,blocked)
        tier_counts=collections.Counter(x["plan"]["tier"] for x in targets)
        theoretical=sum(max(row["imported_rows"],plan_for_city(row)["target"]) for row in gaps)
        print("Capacidade teórica por porte, se houver fonte suficiente:",theoretical,
              "registros; acima de 600 mil:",theoretical>=600000)
        print("Municípios ainda abaixo das metas:",len(targets),
              "| por porte:",dict(tier_counts))
        if not targets:
            print("Sem cidades pendentes para esta configuração.")
            return
        seen=set()
        attempts=success=added=errors=0
        transient_retries=collections.Counter()
        transient_codes={"rede_conexao","rede_timeout","timeout_subprocesso",
                         "fonte_indisponivel","supabase_indisponivel"}
        # No clock-based throttle: each subprocess completes before the next
        # starts. Source/provider errors still stop the session.
        # Limite relativo à execução: não depende do relógio nem da data.
        # O Colab gratuito pode desconectar antes; nenhum bypass de limites.
        deadline=time.monotonic()+MAX_RUN_SECONDS
        print("LIMITE DESTA SESSÃO:",MAX_RUN_SECONDS//3600,
              "horas | até",MAX_DOWNLOADS_PER_SESSION,"tentativas de lote.")
        scheduled=optimized_groups(targets)
        sizes=collections.Counter(len(g) for g in scheduled)
        print("PLANO V6.3: lotes priorizados",len(scheduled),
              "| grupos por quantidade de municípios",dict(sizes),
              "| máximo seguro de novos candidatos por download",3*v5.MAX_NEW_PER_BATCH)
        groups=collections.deque(scheduled)
        while groups:
            group=groups.popleft()
            if attempts>=MAX_DOWNLOADS_PER_SESSION:
                print("PARADA SEGURA: teto de",MAX_DOWNLOADS_PER_SESSION,
                      "tentativas desta sessão atingido; retome em outra sessão.")
                break
            if time.monotonic()>deadline-240:
                print("PARADA SEGURA: duração máxima da sessão (9 horas) aproximada.")
                break
            group=[r for r in group if (r["state"],r["city_key"]) not in seen]
            if not group:continue
            now=dt.datetime.now(dt.timezone.utc)
            stats=v5.current_stats(STAGING_URL,secret)
            if v5.safe_capacity(stats,1)==0:
                print("PARADA: teto nacional/armazenamento atingido.")
                break
            # No deliberate waiting between successfully completed batches.
            # One download at a time; halt on source or storage errors.
            state=group[0]["state"]
            names=[r["city"] for r in group]
            plans={r["city_key"]:r["plan"] for r in group}
            command=[sys.executable,str(WORK/"expand_overture_600k_staging.py"),
                     "--states",state,"--cities",",".join(names),
                     "--plans-json",json.dumps(plans,ensure_ascii=False)]
            env=os.environ.copy()
            env["SUPABASE_URL"]=STAGING_URL
            env["SUPABASE_SERVICE_ROLE_KEY"]=secret
            attempts+=1
            print("LOTE",attempts,"UF",state,"| municípios:",", ".join(names),flush=True)
            try:
                result=subprocess.run(command,env=env,text=True,capture_output=True,
                                      timeout=min(1500,int(max(120,deadline-time.monotonic()-120))))
                output=(result.stdout or "")+"\n"+(result.stderr or "")
                code=None if result.returncode==0 else classify_v5_error(output,result.returncode)
            except subprocess.TimeoutExpired:
                output="";code="timeout_subprocesso"
            # Never echo subprocess stderr (may contain sensitive request headers).
            if code=="batch_largura" and len(group)>1:
                for city in reversed(group):
                    groups.appendleft([city])
                print("LARGURA GEOGRÁFICA: desagrupando",len(group),
                      "municípios para testar separadamente sem download extra.")
                continue
            if code in transient_codes:
                # Retries are allowed only for transient failures. Never
                # automatically retry a source rate limit or access denial.
                signature=tuple((r["state"],r["city_key"]) for r in group)
                used=transient_retries[signature]
                if used<2 and time.monotonic()+270<deadline:
                    pause=(10,30)[used]
                    transient_retries[signature]+=1
                    print("FALHA TEMPORÁRIA:",code,
                          "| nova tentativa",used+1,"de 2 em",pause,
                          "segundos; nenhuma espera fixa nos lotes bem-sucedidos.",
                          flush=True)
                    time.sleep(pause)
                    groups.appendleft(group)
                    continue
            if code:
                errors+=1
                stage=re.findall(r"V5_ERROR_STAGE=([a-z_]{1,40})",output)
                print("ERRO SANITIZADO:",code,
                      "| etapa:",stage[-1] if stage else "não_identificada")
                for row in group:
                    key=row["city_key"]
                    try:
                        previous=len(v5.existing_city(STAGING_URL,secret,state,key,row["plan"]["target"]))
                        v5.record_checkpoint(STAGING_URL,secret,state,key,row["city"],
                                             row["plan"],previous,previous,"error",code)
                    except Exception:
                        print("PARADA: checkpoint indisponível.")
                        return
                if code=="fonte_taxa":
                    next_allowed=dt.datetime.now(dt.timezone.utc)+dt.timedelta(
                        hours=COOLDOWN_HOURS)
                    print("FONTE LIMITOU CONSULTAS. Próxima tentativa conservadora:",
                          next_allowed.strftime("%Y-%m-%d %H:%M UTC"))
                    print("Sem novos downloads ou tentativas automáticas durante a proteção.")
                if stop_for_v5_error(code):
                    print("PARADA GLOBAL:",code,"; não repetir a consulta à fonte agora.")
                    break
                if errors>=3:
                    print("PARADA: 3 falhas; verifique o ambiente sem repetir a fonte.")
                    break
            else:
                success+=1
                for line in output.splitlines():
                    if line.startswith("CITY_RESULT ") or line.startswith("BATCH_SUMMARY "):
                        print(line[:1000])
                        if line.startswith("CITY_RESULT "):
                            try:added+=int(json.loads(line[len("CITY_RESULT "):])["added"])
                            except (ValueError,KeyError,TypeError):pass
            for row in group:
                seen.add((row["state"],row["city_key"]))
        print("RESUMO V6.3: downloads",attempts,"| lotes OK",success,
              "| novas empresas contabilizadas",added,"| erros",errors)
        stats=v5.current_stats(STAGING_URL,secret)
        print("FINAL V6.3: registros",stats["total_companies"],
              "| úteis",stats["useful_companies"],
              "| Overture MiB",round(stats["overture_bytes"]/v5.MIB,2),
              "| banco MiB",round(stats["database_bytes"]/v5.MIB,2))
    except KeyboardInterrupt:
        # Colab pode interromper o kernel, a aba ou o processo sem aviso.
        # Não reinicia nada e não imprime exceções/credenciais.
        print("INTERRUPÇÃO DO COLAB OU DO USUÁRIO: a execução parou.")
        print("Os lotes já gravados permanecem no Supabase de homologação.")
        try:
            stats=v5.current_stats(STAGING_URL,secret)
            print("PARCIAL SALVO:",stats["total_companies"],"registros |",
                  stats["useful_companies"],"úteis.")
        except Exception:
            print("Não foi possível consultar o resumo parcial agora; verifique mais tarde.")
    except Exception as exc:
        print("PARADA SEGURA: ocorreu uma falha de configuração ou serviço:",type(exc).__name__)
    finally:
        secret=""
        print("Chave descartada em memória local. Desconecte o Colab ao terminar.")

# Evita NameError/execução parcial quando o ambiente perdeu a célula 2.
_REQUISITOS_CELULA_2=(
    "POPULATIONS","STAGING_URL","v4","v5","WORK","recent_rate_limit",
    "paged_stage","ordered_queue","plan_for_city","close_groups",
    "classify_v5_error","stop_for_v5_error","optimized_groups","MIN_INTERVAL",
    "COOLDOWN_HOURS","MAX_DOWNLOADS_PER_SESSION","MAX_RUN_SECONDS"
)
_faltantes=[name for name in _REQUISITOS_CELULA_2 if name not in globals()]
if _faltantes:
    print("CONFIGURAÇÃO PENDENTE: execute as células 1 e 2, nesta ordem, antes da célula 3.")
    print("Nenhuma importação foi iniciada. Não cole sua chave secreta aqui.")
elif not hasattr(v5,"safe_capacity") or not hasattr(v5,"current_stats"):
    print("IMPORTADOR INCOMPLETO: execute a célula 2 novamente; nenhuma importação iniciada.")
else:
    ampliar_600k()
